# ModernBERT maize MLM adaptation (Kaggle)

Run this notebook on Kaggle with internet enabled and version 3 of `gurveersinghvirk/modernflorabert-base` attached as a notebook input. The code discovers the mounted input first and uses KaggleHub only as a fallback.

The repository branch, plant-pretrained checkpoint, tokenizer, and current MLM recipe match the Colab maize-MLM notebook. The output is written under `/kaggle/working`, not into the read-only input dataset. No tokenizer retraining or random ModernBERT initialization is permitted.

W&B authentication is optional only in the sense that its key can be supplied at runtime; run the W&B cell before training so the repository Trainer does not fail during its first log event.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


kaggle_input_root = Path(
    os.environ.get('KAGGLE_INPUT_ROOT', '/kaggle/input')
).expanduser()
run_root = Path(
    os.environ.get('FLORABERT_RUN_ROOT', '/kaggle/working/florabert_runs')
).expanduser()
repo_dir = Path(
    os.environ.get('FLORABERT_REPO_DIR', '/kaggle/working/florabert')
).expanduser()
repo_url = os.environ.get(
    'FLORABERT_REPO_URL',
    'https://github.com/gurveersinghvirk/florabert.git',
)
repo_ref = os.environ.get(
    'FLORABERT_REPO_REF',
    'feat/modernbert-maize-mlm-ablation',
)
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '').strip()

if not (repo_dir / '.git').is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f'{repo_dir} exists but is not an empty git checkout'
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--branch', repo_ref, '--depth', '1', repo_url, str(repo_dir)],
        check=True,
    )

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repo_dir,
    text=True,
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; expected {expected_commit}'
    )

run_root.mkdir(parents=True, exist_ok=True)
data_root = run_root / 'data'
hf_maize_dir = data_root / 'maize-promoter-sequences'
kaggle_workspace = run_root / 'kaggle-modernflorabert-base-v3'
kaggle_workspace.mkdir(parents=True, exist_ok=True)
plant_checkpoint = kaggle_workspace / 'plant-checkpoint-6000'
plant_tokenizer = kaggle_workspace / 'modernbert-tokenizer'
maize_lm_output = (
    run_root / 'models' / 'transformer' / 'language-model-modernbert-maize'
)

for path in [data_root, hf_maize_dir, maize_lm_output]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))

print('Kaggle input root:', kaggle_input_root)
print('Repo:', repo_dir)
print('Repo ref:', repo_ref)
print('Repo commit:', actual_commit)
print('Run root:', run_root)
print('Maize data directory:', hf_maize_dir)
print('Kaggle working artifact directory:', kaggle_workspace)
print('Maize MLM output directory:', maize_lm_output)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'requirements.txt'),
    ],
    cwd=repo_dir,
)

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'kagglehub>=1.0',
        'wandb>=0.19',
    ]
)

import importlib.metadata as importlib_metadata
import torch

for package_name in [
    'torch',
    'transformers',
    'datasets',
    'accelerate',
    'huggingface-hub',
    'kagglehub',
    'wandb',
]:
    try:
        print(package_name, importlib_metadata.version(package_name))
    except importlib_metadata.PackageNotFoundError:
        print(package_name, 'not found')

cuda_count = torch.cuda.device_count()
print('CUDA device count:', cuda_count)
if cuda_count:
    for device_idx in range(cuda_count):
        print(f'CUDA {device_idx}: {torch.cuda.get_device_name(device_idx)}')
else:
    raise RuntimeError('A Kaggle GPU runtime is required for practical MLM training.')

In [ ]:
from huggingface_hub import hf_hub_download


def runtime_secret(name):
    value = os.environ.get(name)
    return value.strip() if value and value.strip() else None


hf_token = runtime_secret('HF_TOKEN') or runtime_secret('HUGGINGFACE_TOKEN')
if hf_token:
    from huggingface_hub import login as hf_login

    hf_login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authentication loaded from an environment variable.')
else:
    print('No HF token supplied; the requested maize dataset is public.')


kaggle_token = runtime_secret('KAGGLE_API_TOKEN')
if kaggle_token:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Kaggle API token found in the runtime environment.')
else:
    print(
        'No Kaggle API token in the environment; attached public inputs '
        'and KaggleHub cached credentials will be used.'
    )

import kagglehub
print('KaggleHub is ready.')

In [ ]:
import wandb


wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    from getpass import getpass

    wandb_key = getpass(
        'Weights & Biases API key (hidden input; not saved in this notebook): '
    ).strip()

if not wandb_key:
    raise RuntimeError('No W&B API key was entered.')

os.environ['WANDB_API_KEY'] = wandb_key
os.environ.setdefault(
    'WANDB_PROJECT',
    'florabert-modernbert-maize-ablation',
)
wandb.login(relogin=True)
del wandb_key

print('W&B authentication is ready.')
print('W&B project:', os.environ['WANDB_PROJECT'])

In [ ]:
hf_dataset = 'Gurveer05/maize-promoter-sequences'
maize_files = ['all_seqs_train.txt', 'all_seqs_test.txt']


def count_nonempty_lines(path):
    count = 0
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                count += 1
    return count


maize_paths = {}
maize_counts = {}
for filename in maize_files:
    hf_hub_download(
        repo_id=hf_dataset,
        filename=filename,
        repo_type='dataset',
        local_dir=str(hf_maize_dir),
        token=hf_token,
    )

    path = hf_maize_dir / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty maize file: {path}')

    maize_paths[filename] = path
    maize_counts[filename] = count_nonempty_lines(path)
    if maize_counts[filename] == 0:
        raise ValueError(f'Maize file has no non-empty sequences: {path}')

    print(
        filename,
        '->',
        path,
        'bytes=',
        path.stat().st_size,
        'sequences=',
        maize_counts[filename],
    )

assert maize_counts['all_seqs_train.txt'] > 0
assert maize_counts['all_seqs_test.txt'] > 0

In [ ]:
# Prefer the read-only version-3 dataset attached under /kaggle/input.
# This avoids an unnecessary archive download and makes the input source
# visible before training starts.
kaggle_dataset = 'gurveersinghvirk/modernflorabert-base/versions/3'

if kaggle_input_root.is_dir():
    input_dirs = sorted(
        path for path in kaggle_input_root.iterdir() if path.is_dir()
    )
    print('Mounted Kaggle input directories:')
    for path in input_dirs:
        print(' ', path)

weight_files = sorted(
    path
    for path in kaggle_input_root.rglob('model.safetensors')
    if 'modernbert' in str(path).lower()
    and 'language-model' in str(path).lower()
) if kaggle_input_root.is_dir() else []
tokenizer_files = sorted(
    path
    for path in kaggle_input_root.rglob('tokenizer.json')
    if 'modernbert' in str(path).lower()
) if kaggle_input_root.is_dir() else []

print('ModernBERT weight candidates:')
for path in weight_files:
    print(' ', path)
print('ModernBERT tokenizer candidates:')
for path in tokenizer_files:
    print(' ', path)

if weight_files and tokenizer_files:
    plant_checkpoint = weight_files[0].parent
    plant_tokenizer = tokenizer_files[0].parent
    print('Using attached version-3 input.')
else:
    print('Attached version-3 artifacts were not found; using KaggleHub fallback.')
    kagglehub_stage_dir = kaggle_workspace / '.kagglehub-files'
    kagglehub_stage_dir.mkdir(parents=True, exist_ok=True)

    kaggle_specs = [
        (
            'florabert/models/transformer/language-model-modernbert/config.json',
            plant_checkpoint,
        ),
        (
            'florabert/models/transformer/language-model-modernbert/model.safetensors',
            plant_checkpoint,
        ),
        (
            'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer.json',
            plant_tokenizer,
        ),
        (
            'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer_config.json',
            plant_tokenizer,
        ),
        (
            'florabert/models/modernbert-byte-level-bpe-tokenizer/special_tokens_map.json',
            plant_tokenizer,
        ),
    ]

    def download_kaggle_file(remote_name, destination_dir):
        destination_dir = Path(destination_dir)
        destination_dir.mkdir(parents=True, exist_ok=True)
        local_path = destination_dir / Path(remote_name).name

        downloaded_path = Path(
            kagglehub.dataset_download(
                kaggle_dataset,
                path=remote_name,
                output_dir=str(kagglehub_stage_dir),
            )
        )
        candidates = [downloaded_path, kagglehub_stage_dir / remote_name]
        if downloaded_path.is_dir():
            candidates.extend(downloaded_path.rglob(Path(remote_name).name))

        source_path = next(
            (path for path in candidates if path.is_file()),
            None,
        )
        if source_path is None or source_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'KaggleHub did not produce {remote_name}; returned {downloaded_path}'
            )

        if source_path.resolve() != local_path.resolve():
            shutil.copy2(source_path, local_path)
        return local_path

    for remote_name, destination_dir in kaggle_specs:
        download_kaggle_file(remote_name, destination_dir)

print('Resolved plant checkpoint:', plant_checkpoint)
print('Resolved plant tokenizer:', plant_tokenizer)

In [ ]:
from transformers import AutoConfig, PreTrainedTokenizerFast

from module.florabert import config as flora_config
from module.florabert import transformers as flora_transformers
from module.florabert import utils as flora_utils


plant_config = AutoConfig.from_pretrained(
    str(plant_checkpoint),
    local_files_only=True,
)
plant_tokenizer_obj = PreTrainedTokenizerFast.from_pretrained(
    str(plant_tokenizer),
    local_files_only=True,
)

print('Selected pretrained checkpoint:', plant_checkpoint)
print('Selected tokenizer:', plant_tokenizer)
print('Model type:', plant_config.model_type)
print('Architectures:', plant_config.architectures)
print('Checkpoint vocab size:', plant_config.vocab_size)
print('Tokenizer vocab size:', len(plant_tokenizer_obj))
print('Model max positions:', plant_config.max_position_embeddings)

assert plant_config.model_type == 'modernbert'
assert 'ModernBertForMaskedLM' in (plant_config.architectures or [])
assert plant_config.vocab_size == len(plant_tokenizer_obj), (
    'Refusing to resize or retrain the plant tokenizer: checkpoint vocab '
    f'{plant_config.vocab_size} != tokenizer vocab {len(plant_tokenizer_obj)}'
)
assert (plant_checkpoint / 'model.safetensors').is_file()

flora_config.reload_settings()
lm_settings = flora_utils.get_model_settings(
    flora_config.settings,
    model_name='modernbert-lm',
)
expected_positions = lm_settings['max_tokenized_len'] + 2
assert plant_config.max_position_embeddings == expected_positions

_, loaded_tokenizer, plant_lm = flora_transformers.load_model(
    'modernbert-lm',
    str(plant_tokenizer),
    pretrained_model=str(plant_checkpoint),
    **lm_settings,
)

total_params = flora_utils.count_model_parameters(
    plant_lm,
    trainable_only=False,
)
print('Verified plant checkpoint weights loaded.')
print('Loaded parameter count:', total_params)
assert len(loaded_tokenizer) == plant_config.vocab_size

smoke_inputs = loaded_tokenizer(
    'ACGTACGTACGTTTTAAACCCGGG',
    return_tensors='pt',
    max_length=loaded_tokenizer.model_max_length,
    truncation=True,
    padding='max_length',
)
plant_lm.eval()
with torch.no_grad():
    smoke_outputs = plant_lm(**smoke_inputs)

assert smoke_outputs.logits.ndim == 3
assert torch.isfinite(smoke_outputs.logits).all()
print('One MLM forward pass:', tuple(smoke_outputs.logits.shape))

del plant_lm, loaded_tokenizer, smoke_outputs, smoke_inputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
pretrain_settings = dict(flora_config.settings['training']['pretrain'])
mlm_learning_rate = (
    float(os.environ['FLORABERT_MLM_LEARNING_RATE'])
    if os.environ.get('FLORABERT_MLM_LEARNING_RATE')
    else None
)
mlm_epochs = (
    int(os.environ['FLORABERT_MLM_EPOCHS'])
    if os.environ.get('FLORABERT_MLM_EPOCHS')
    else None
)
n_workers = int(os.environ.get('FLORABERT_DATA_WORKERS', '2'))
force_rerun = env_bool('FLORABERT_FORCE_RERUN', False)
mlm_resume_from = (
    Path(os.environ['FLORABERT_MLM_RESUME_FROM']).expanduser()
    if os.environ.get('FLORABERT_MLM_RESUME_FROM')
    else None
)

print('Current repo MLM settings:', pretrain_settings)
print('Effective learning rate:', mlm_learning_rate or pretrain_settings['learning_rate'])
print('Effective epochs:', mlm_epochs or pretrain_settings['num_train_epochs'])
print('MLM resume checkpoint:', mlm_resume_from or 'none')
print('CUDA devices:', cuda_count)

In [ ]:
def launch_repo_script(relative_script, arguments):
    script_path = repo_dir / relative_script

    if cuda_count > 1:
        accelerate_exe = shutil.which('accelerate')
        if accelerate_exe:
            command = [
                accelerate_exe,
                'launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
        else:
            command = [
                sys.executable,
                '-m',
                'accelerate.commands.launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
    else:
        command = [sys.executable, '-u', str(script_path), *map(str, arguments)]

    environment = os.environ.copy()
    environment['PYTHONPATH'] = (
        str(repo_dir) + os.pathsep + environment.get('PYTHONPATH', '')
    )
    environment['PYTHONUNBUFFERED'] = '1'

    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(repo_dir),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

In [ ]:
mlm_source = mlm_resume_from or plant_checkpoint
if not (mlm_source / 'config.json').is_file():
    raise FileNotFoundError(
        f'MLM source checkpoint is missing config.json: {mlm_source}'
    )
if not any(
    path.is_file()
    for pattern in ('*.safetensors', '*.bin', '*.safetensors.index.json', '*.bin.index.json')
    for path in mlm_source.glob(pattern)
):
    raise FileNotFoundError(
        f'MLM source checkpoint has no model weights: {mlm_source}'
    )

print('Resolved maize train:', maize_paths['all_seqs_train.txt'])
print('Resolved maize test:', maize_paths['all_seqs_test.txt'])
print('Maize train sequences:', maize_counts['all_seqs_train.txt'])
print('Maize test sequences:', maize_counts['all_seqs_test.txt'])
print('Pretrained MLM checkpoint being loaded:', mlm_source)
print('Maize MLM output:', maize_lm_output)

maize_output_config = maize_lm_output / 'config.json'
if (
    maize_output_config.is_file()
    and not force_rerun
    and mlm_resume_from is None
):
    print(
        'Maize MLM output already exists; set FLORABERT_FORCE_RERUN=1 '
        'to retrain:',
        maize_lm_output,
    )
else:
    mlm_arguments = [
        '--model-name',
        'modernbert-lm',
        '--data-dir',
        str(hf_maize_dir),
        '--train-data',
        'all_seqs_train.txt',
        '--test-data',
        'all_seqs_test.txt',
        '--tokenizer-dir',
        str(plant_tokenizer),
        '--output-dir',
        str(maize_lm_output),
        '--precision',
        'fp16',
        '--n-workers',
        str(n_workers),
        '--pretrained-model',
        str(mlm_source),
    ]

    if mlm_resume_from is not None:
        mlm_arguments.extend(['--resume-from-checkpoint', str(mlm_resume_from)])
    if mlm_learning_rate is not None:
        mlm_arguments.extend(['--learning-rate', str(mlm_learning_rate)])
    if mlm_epochs is not None:
        mlm_arguments.extend(['--num-train-epochs', str(mlm_epochs)])

    launch_repo_script(
        Path('scripts/1-modeling/pretrain.py'),
        mlm_arguments,
    )

In [ ]:
assert (maize_lm_output / 'config.json').is_file()
assert any(
    path.is_file()
    for pattern in ('*.safetensors', '*.bin', '*.safetensors.index.json', '*.bin.index.json')
    for path in maize_lm_output.glob(pattern)
)

print('Maize-adapted ModernBERT checkpoint is ready:', maize_lm_output)
print('The checkpoint is available under /kaggle/working for downstream use.')